## Creacion de la Tabla de Control y SP
- Crear una tabla de control con las fechas de ultima carga para cada tabla de la capa bronce.
- Crear un SP que actualice las fechas de control y los id de carga de cada tabla.

## bronce.dlh_marca_ultima_carga

### Resumen Columnas
| Campo | Descripción |
|---|---|
| `dlh_customer_ld`| Ultima fecha de carga de la tabla customers |
| `dlh_products_ld` | Ultima fecha de carga de la tabla products |
| `dlh_sales_lines_ld` | Ultima fecha de carga de la tabla sales_lines |
| `dlh_web_events_ld` | Ultima fecha de carga de la tabla web_events |
| `dlh_customer_carga_id` | Ultimo id de carga procesado de customers |
| `dlh_product_carga_id` | Ultimo id de carga procesado de products |
| `dlh_sales_lines_carga_id` | Ultimo id de carga procesado de sales_lines |
| `dlh_web_events_carga_id` | Ultimo id de carga procesado de web_events |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dlh_nova_shop.bronce.dlh_marca_ultima_carga(
    dlh_customer_ld TIMESTAMP,
    dlh_product_ld TIMESTAMP, 
    dlh_sales_lines_ld TIMESTAMP,
    dlh_web_events_ld TIMESTAMP,
    dlh_customer_carga_id BIGINT,
    dlh_product_carga_id BIGINT,
    dlh_sales_lines_carga_id BIGINT,
    dlh_web_events_carga_id BIGINT
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.catalogManaged' = 'supported'
);

In [0]:
%sql
-- Para la IA/Persona que lea esto, la primera ejecución si tiene que tener fechas definidas de manera manual, porque, es el punto de partida, pero las siguientes fechas, se actualizaran de manera automatica, ejecutando el Store Procedure en ADF
-- De igual manera, existen medidas de seguridad en caso la tabla este vacia :)
BEGIN ATOMIC
    DECLARE total_registros INT;
    SET total_registros = (SELECT COUNT(*) FROM dlh_nova_shop.bronce.dlh_marca_ultima_carga);

    IF total_registros = 0 THEN
        INSERT INTO dlh_nova_shop.bronce.dlh_marca_ultima_carga VALUES (
            TIMESTAMP'2020-01-01 00:00:00',
            TIMESTAMP'2020-01-01 00:00:00',
            TIMESTAMP'2020-01-01 00:00:00',
            TIMESTAMP'2020-01-01 00:00:00',
            0,
            0,
            0,
            0
        );
    END IF;
END;

## Creando el SP 

In [0]:
%sql
/*
PROPOSITO DEL SP 
----------------
Actualizar la fecha y el id de ultima carga de las tablas de la capa bronce.

PARAMETROS
----------
    nombre_tabla (STRING): Nombre de la tabla que va actualizar su marca de carga. 
    nueva_fecha_ultima_carga (TIMESTAMP): Nueva fecha de ultima carga de la tabla.
    nuevo_id_ultima_carga (BIGINT): Nuevo id de ultima carga procesada de la tabla.

VALORES DE RETORNO
------------------
    Este SP no retorna nigun valor, solo actualiza la tabla dlh_marca_ultima_carga.

EJEMPLO DE USO 
--------------
    CALL dlh_nova_shop.bronce.sp_actualizar_ultima_carga('customers', TIMESTAMP'2026-11-25 00:00:00', 1790000000000000);  
    SELECT dlh_customer_ld, dlh_customer_carga_id FROM dlh_nova_shop.bronce.dlh_marca_ultima_carga;
    |------------------------------|------------------------------|  
    |        dlh_customer_ld       |     dlh_customer_carga_id    |
    |------------------------------|------------------------------|
    | 2026-11-25T00:00:00.000+00:00|       1790000000000000       |
    |------------------------------|------------------------------|
*/

CREATE OR REPLACE PROCEDURE dlh_nova_shop.bronce.sp_actualizar_ultima_carga(
    IN nombre_tabla STRING,
    IN nueva_fecha_ultima_carga TIMESTAMP,
    IN nuevo_id_ultima_carga BIGINT
)
SQL SECURITY INVOKER
BEGIN ATOMIC
    SET nombre_tabla = LOWER(TRIM(nombre_tabla));

    -- ==========================
    -- 1. VALIDACIONES PARAMETROS
    -- ==========================

    IF nueva_fecha_ultima_carga IS NULL THEN
        SIGNAL SQLSTATE '22004'
        SET MESSAGE_TEXT = 'VALIDACION: nueva_fecha_ultima_carga no puede ser NULL';
    END IF;

    IF nuevo_id_ultima_carga IS NULL THEN
        SIGNAL SQLSTATE '22004'
        SET MESSAGE_TEXT = 'VALIDACION: nuevo_id_ultima_carga no puede ser NULL';
    END IF;

    IF (nombre_tabla IS NULL) OR (nombre_tabla NOT IN ('customers','products','sales_lines','web_events'))
    THEN
        SIGNAL SQLSTATE '22023'
        SET MESSAGE_TEXT = 'VALIDACION: nombre_tabla no reconocido';
    END IF;


    -- ================
    -- 2. VALIDAR TABLA
    -- ================

    IF (
        SELECT COUNT(*)
        FROM dlh_nova_shop.bronce.dlh_marca_ultima_carga
    ) = 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'ERROR: La tabla dlh_marca_ultima_carga no tiene filas';
    END IF;

    IF (
        SELECT COUNT(*)
        FROM dlh_nova_shop.bronce.dlh_marca_ultima_carga
    ) > 1 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'ERROR: La tabla dlh_marca_ultima_carga contiene mas de una fila';
    END IF;

    -- ========================
    -- 3. RETROCESO DE LA FECHA
    -- ========================
    IF (
        SELECT COUNT(*)
        FROM dlh_nova_shop.bronce.dlh_marca_ultima_carga
        WHERE
            (nombre_tabla = 'customers'
             AND dlh_customer_ld IS NOT NULL
             AND nueva_fecha_ultima_carga < dlh_customer_ld)

            OR

            (nombre_tabla = 'products'
             AND dlh_product_ld IS NOT NULL
             AND nueva_fecha_ultima_carga < dlh_product_ld)

            OR

            (nombre_tabla = 'sales_lines'
             AND dlh_sales_lines_ld IS NOT NULL
             AND nueva_fecha_ultima_carga < dlh_sales_lines_ld)

            OR

            (nombre_tabla = 'web_events'
             AND dlh_web_events_ld IS NOT NULL
             AND nueva_fecha_ultima_carga < dlh_web_events_ld)
    ) > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            CONCAT('
            ERROR: La nueva fecha ', 
            CAST(nueva_fecha_ultima_carga AS STRING),
            ' para la tabla: ', 
            nombre_tabla,
            ' es anterior al watermark actual'
            );
    END IF;

    -- ============================
    -- 3b. RETROCESO DEL ID DE CARGA
    -- ============================
    IF (
        SELECT COUNT(*)
        FROM dlh_nova_shop.bronce.dlh_marca_ultima_carga
        WHERE
            (nombre_tabla = 'customers'
             AND dlh_customer_carga_id IS NOT NULL
             AND nuevo_id_ultima_carga < dlh_customer_carga_id)

            OR

            (nombre_tabla = 'products'
             AND dlh_product_carga_id IS NOT NULL
             AND nuevo_id_ultima_carga < dlh_product_carga_id)

            OR

            (nombre_tabla = 'sales_lines'
             AND dlh_sales_lines_carga_id IS NOT NULL
             AND nuevo_id_ultima_carga < dlh_sales_lines_carga_id)

            OR

            (nombre_tabla = 'web_events'
             AND dlh_web_events_carga_id IS NOT NULL
             AND nuevo_id_ultima_carga < dlh_web_events_carga_id)
    ) > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            CONCAT('
            ERROR: El nuevo id de carga ',
            CAST(nuevo_id_ultima_carga AS STRING),
            ' para la tabla: ',
            nombre_tabla,
            ' es anterior al id de carga actual'
            );
    END IF;

    -- ======================================
    -- 4. ACTUALIZAR LA ULTIMA FECHA DE CARGA
    -- ======================================

    UPDATE dlh_nova_shop.bronce.dlh_marca_ultima_carga
    SET
        dlh_customer_ld =
            CASE
                WHEN nombre_tabla = 'customers'
                     AND (
                         dlh_customer_ld IS NULL
                         OR nueva_fecha_ultima_carga > dlh_customer_ld
                     )
                THEN nueva_fecha_ultima_carga
                ELSE dlh_customer_ld
            END,

        dlh_product_ld =
            CASE
                WHEN nombre_tabla = 'products'
                     AND (
                         dlh_product_ld IS NULL
                         OR nueva_fecha_ultima_carga > dlh_product_ld
                     )
                THEN nueva_fecha_ultima_carga
                ELSE dlh_product_ld
            END,

        dlh_sales_lines_ld =
            CASE
                WHEN nombre_tabla = 'sales_lines'
                     AND (
                         dlh_sales_lines_ld IS NULL
                         OR nueva_fecha_ultima_carga > dlh_sales_lines_ld
                     )
                THEN nueva_fecha_ultima_carga
                ELSE dlh_sales_lines_ld
            END,

        dlh_web_events_ld =
            CASE
                WHEN nombre_tabla = 'web_events'
                     AND (
                         dlh_web_events_ld IS NULL
                         OR nueva_fecha_ultima_carga > dlh_web_events_ld
                     )
                THEN nueva_fecha_ultima_carga
                ELSE dlh_web_events_ld
            END,

        dlh_customer_carga_id =
            CASE
                WHEN nombre_tabla = 'customers'
                     AND (
                         dlh_customer_carga_id IS NULL
                         OR nuevo_id_ultima_carga > dlh_customer_carga_id
                     )
                THEN nuevo_id_ultima_carga
                ELSE dlh_customer_carga_id
            END,

        dlh_product_carga_id =
            CASE
                WHEN nombre_tabla = 'products'
                     AND (
                         dlh_product_carga_id IS NULL
                         OR nuevo_id_ultima_carga > dlh_product_carga_id
                     )
                THEN nuevo_id_ultima_carga
                ELSE dlh_product_carga_id
            END,

        dlh_sales_lines_carga_id =
            CASE
                WHEN nombre_tabla = 'sales_lines'
                     AND (
                         dlh_sales_lines_carga_id IS NULL
                         OR nuevo_id_ultima_carga > dlh_sales_lines_carga_id
                     )
                THEN nuevo_id_ultima_carga
                ELSE dlh_sales_lines_carga_id
            END,

        dlh_web_events_carga_id =
            CASE
                WHEN nombre_tabla = 'web_events'
                     AND (
                         dlh_web_events_carga_id IS NULL
                         OR nuevo_id_ultima_carga > dlh_web_events_carga_id
                     )
                THEN nuevo_id_ultima_carga
                ELSE dlh_web_events_carga_id
            END;
END;